# 05 — Full Pipeline: End-to-End Allocation

This is the **master notebook** — runs the complete workflow in one go:

```
Download → Clean → Estimate μ & Σ → Optimize → Capital Allocate → Report
```

Customize `config.py` to change assets, risk aversion, and constraints.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.makedirs('../results', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.data_fetcher.fetcher import download_prices, compute_returns, describe_returns
from src.estimators.returns import historical_mean
from src.estimators.covariance import estimate_covariance
from src.optimizers.markowitz import minimum_variance, maximum_sharpe, maximum_utility, efficient_frontier
from src.optimizers.risk_parity import risk_parity, risk_contribution_report
from src.optimizers.cvar_optimizer import cvar_optimization, portfolio_cvar
from src.risk.metrics import portfolio_report, optimal_risky_weight, capital_allocation_line
from src.utils.plotting import (
    plot_efficient_frontier, plot_weights_bar, plot_risk_contributions,
    plot_cumulative_returns, plot_correlation_heatmap
)
from config import *
print('All modules loaded ✓')

## Step 1: Data

In [ ]:
prices  = download_prices(TICKERS, START_DATE, interval=PRICE_FREQUENCY, cache_dir='../data/')
returns = compute_returns(prices, method=RETURN_TYPE)
print(describe_returns(returns).to_string())

## Step 2: Estimation

In [ ]:
mu  = historical_mean(returns)
cov = estimate_covariance(returns, method=COV_METHOD)
print(f'μ estimated ({MU_METHOD}), Σ estimated ({COV_METHOD}) ✓')
print(f'Expected returns (annual):')
for t, v in (mu*100).items():
    print(f'  {t}: {v:.2f}%')

## Step 3: Portfolio Optimization

In [ ]:
p_minvar = minimum_variance(mu, cov, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)
p_sharpe = maximum_sharpe(mu, cov, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)
p_util   = maximum_utility(mu, cov, RISK_AVERSION, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)
p_rp     = risk_parity(mu, cov, RISK_FREE_RATE_ANNUAL)
p_cvar   = cvar_optimization(returns, mu, cov, CVAR_ALPHA, RISK_FREE_RATE_ANNUAL, MIN_WEIGHT, MAX_WEIGHT)

portfolios = {
    'Min Variance': p_minvar,
    'Max Sharpe': p_sharpe,
    f'Max Utility (A={RISK_AVERSION})': p_util,
    'Risk Parity': p_rp,
    f'Min CVaR ({int((1-CVAR_ALPHA)*100)}%)': p_cvar,
}

for name, p in portfolios.items():
    print(p)

## Step 4: Capital Allocation

In [ ]:
# Use Tangency Portfolio for capital allocation
tp = p_sharpe
y_star = optimal_risky_weight(tp.expected_return, tp.volatility, RISK_FREE_RATE_ANNUAL, RISK_AVERSION)

print('='*55)
print('CAPITAL ALLOCATION DECISION')
print('='*55)
print(f'Tangency Portfolio:  E[r]={tp.expected_return*100:.2f}%, σ={tp.volatility*100:.2f}%, SR={tp.sharpe_ratio:.2f}')
print(f'Risk-Free Rate:      {RISK_FREE_RATE_ANNUAL*100:.2f}%')
print(f'Risk Aversion (A):   {RISK_AVERSION}')
print(f'')
print(f'Formula: y* = (E[rp]-rf) / (A·σp²)')
print(f'         y* = ({tp.expected_return*100:.2f}% - {RISK_FREE_RATE_ANNUAL*100:.2f}%) / ({RISK_AVERSION} × {tp.volatility*100:.2f}%²)')
print(f'         y* = {y_star*100:.1f}%')
print()
print(f'ALLOCATION:')
print(f'  {y_star*100:.1f}% Risky Portfolio (Tangency)')
print(f'  {(1-y_star)*100:.1f}% Risk-Free Asset')
print()
print('Within the Risky Portfolio:')
for asset, w in (tp.weights * y_star * 100).sort_values(ascending=False).items():
    print(f'  {asset:6s}: {w:5.1f}%  (→ {tp.weights[asset]*100:.1f}% of risky portion)')

## Step 5: Visualizations

In [ ]:
print('Computing frontier...')
frontier = efficient_frontier(mu, cov, RISK_FREE_RATE_ANNUAL, N_FRONTIER_POINTS, MIN_WEIGHT, MAX_WEIGHT)

fig = plot_efficient_frontier(frontier, portfolios, RISK_FREE_RATE_ANNUAL, '../results/05_frontier.png')
plt.show()

fig = plot_weights_bar(portfolios, '../results/05_weights.png')
plt.show()

fig = plot_cumulative_returns({n: p.weights for n,p in portfolios.items()},
                               returns=returns,
                               risk_free_rate_monthly=RISK_FREE_RATE_MONTHLY,
                               save_path='../results/05_cumulative.png')
plt.show()

## Step 6: Risk Report Summary

In [ ]:
print('\n' + '='*70)
print('COMPREHENSIVE RISK/PERFORMANCE REPORT')
print('='*70)

rows = []
for name, p in portfolios.items():
    r = portfolio_report(p.weights, returns, mu, cov, RISK_FREE_RATE_ANNUAL, RISK_AVERSION)
    r['Strategy'] = name
    rows.append(r)

df = pd.DataFrame(rows).set_index('Strategy')
pd.set_option('display.max_columns', None)
display(df.T)

df.T.to_csv('../results/05_report.csv')
print('\nReport saved to ../results/05_report.csv ✓')